# Kinase LID Redesign

**BioPipelines example** — de novo backbone design of the adenylate kinase LID domain using RFdiffusion, followed by ProteinMPNN sequence design and AlphaFold2 refolding. Candidates are filtered by RMSD of the fixed regions and pLDDT, then visualised in PyMOL.

[![Documentation](https://img.shields.io/badge/docs-readthedocs-blue)](https://biopipelines.readthedocs.io/en/latest/)
[![Preprint](https://img.shields.io/badge/preprint-bioRxiv-B31B1B)](https://www.biorxiv.org/content/10.64898/2026.03.11.711024v1)

In [ ]:
# Cell 1: Install BioPipelines and micromamba
!git clone https://github.com/locbp-uzh/biopipelines
%cd biopipelines
!pip install -e ".[all]"
!curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj -C /usr/local bin/micromamba
!micromamba create -f Environments/biopipelines.yaml -y

In [ ]:
# Cell 2: Install tools
from biopipelines.pipeline import *
from biopipelines.rfdiffusion import RFdiffusion
from biopipelines.protein_mpnn import ProteinMPNN
from biopipelines.alphafold import AlphaFold
from biopipelines.pymol import PyMOL

with Pipeline(project="Setup", job="InstallTools"):
    RFdiffusion.install()
    ProteinMPNN.install()
    AlphaFold.install()
    PyMOL.install()

## Cell 3: Kinase LID Redesign Pipeline

Takes adenylate kinase (PDB: 4AKE) and redesigns the LID domain (residues 118–160, replaced with a new 50–70 residue loop).
The top 3 designs with RMSD < 1.5 Å on the fixed scaffold and highest pLDDT are coloured and loaded into PyMOL.

In [ ]:
# Cell 3: Pipeline
from biopipelines.pipeline import *
from biopipelines.rfdiffusion import RFdiffusion
from biopipelines.protein_mpnn import ProteinMPNN
from biopipelines.alphafold import AlphaFold
from biopipelines.conformational_change import ConformationalChange
from biopipelines.panda import Panda
from biopipelines.pymol import PyMOL

with Pipeline(project="AdenylateKinase", job="LID_Redesign"):
    kinase = PDB("4AKE")
    backbones = RFdiffusion(pdb=kinase,
                            contigs='A1-117/50-70/A161-214',
                            num_designs=10)
    sequences = ProteinMPNN(structures=backbones,
                            num_sequences=2,
                            redesigned=backbones.tables.structures.designed)
    refolded = AlphaFold(proteins=sequences)
    conf_change = ConformationalChange(reference_structures=kinase,
                                       target_structures=refolded,
                                       selection=backbones.tables.structures.fixed)
    top3 = Panda(tables=[refolded.tables.confidence,
                          conf_change.tables.changes],
                 operations=[Panda.merge(),
                              Panda.filter("RMSD < 1.5"),
                              Panda.sort("plddt", ascending=False),
                              Panda.head(3)],
                 pool=refolded)
top3